# 02 · AF 데이터셋 기반 그림·표·헤드라인 수치  ★핵심

`map_exports/e10/{Ref,HalfSC,SC}/*_Map_Summary.json` (모델당 ~760 kB, git 추적) 만 있으면 된다. **클론만으로 재현 가능한 유일한 묶음**이라 리뷰어·2차 검증이 실제로 돌릴 수 있는 부분이다.

| 산출 | 내용 | 스크립트 |
|---|---|---|
| Fig 5 | 보정 형태별 수렴 (3패널) | `run_manuscript_figs78.py` |
| Fig 6 | 전달 플랜 표본 수 감축 맵 | 〃 |
| Fig 10 | SC 보정 검증 parity + 오차분포 | `pipeline` |
| Table 3 | 다섯 보정 형태 비교 (10-seed) | `run_form_study.py` |
| 헤드라인 | 무보정 → 보정 후 wMAE, 후보 풀 | 아래 감사 셀 |


In [ ]:
import os, sys, runpy, time

# 이 노트북이 있는 곳의 부모 = mlxperPJT/JEET
JEET = os.path.abspath(os.path.join(os.getcwd(), '..'))
TOOLS = os.path.abspath(os.path.join(JEET, '..', '..', 'tools'))
if TOOLS not in sys.path:
    sys.path.insert(0, TOOLS)

# 그림 출력 폴더 — 배포 레포에서는 여기만 바꾸면 된다
os.environ.setdefault('JEET_FIGDIR',
                      os.path.abspath(os.path.join(JEET, 'fig_out')))
os.makedirs(os.environ['JEET_FIGDIR'], exist_ok=True)
print('JEET  :', JEET)
print('FIGDIR:', os.environ['JEET_FIGDIR'])


In [ ]:
def run(script, note=''):
    """run_*.py 를 __main__ 으로 실행하고 소요 시간을 찍는다."""
    p = os.path.join(JEET, script)
    assert os.path.exists(p), p
    t0 = time.time()
    print('>>>', script, note)
    runpy.run_path(p, run_name='__main__')
    print('    %.1f s' % (time.time() - t0))


## Fig 5 · Fig 6 — 수렴과 ablation

10-seed 평균이 들어가 수 분 걸린다.


In [ ]:
run('run_manuscript_figs78.py', '(Fig 5, Fig 6)')


## Fig 10 — SC 검증 parity

적합은 표본 후보 풀, **채점·작도는 전 부하점(96)** — 본문 §5.3 과 같은 모집단이다 (`eval_all_load_points=True`).


In [ ]:
from jeet_acloss_rbf.pipeline import AcLossPipeline

pl = AcLossPipeline()
out = os.path.join(os.environ['JEET_FIGDIR'],
                   'RBF_correction_validation_SC.png')
print(pl.make_validation_figure('SC', out,
                                eval_all_load_points=True))


## Table 3 — 보정 형태 비교


In [ ]:
from jeet_acloss_rbf import form_study

res = form_study.run_form_study(AcLossPipeline(), n_seeds=10)
ORDER = ['kcc', 'perspeed', 'tps3d', 'scalar', 'exponent']
NAME = {'kcc': 'Polynomial k_cc', 'perspeed': 'Per-speed quad',
        'tps3d': 'Non-separable 3-D TPS', 'scalar': 'Scalar separable',
        'exponent': 'Exponent separable (adopted)'}
print('%-30s %-14s %-14s %-14s' % ('form', 'Ref', 'HalfSC', 'SC'))
for fm in ORDER:
    cells = []
    for sc in ('Ref', 'HalfSC', 'SC'):
        pc = res['scales'][sc]['placement']
        cells.append('%.1f (%.1f)' % (
            pc['structured']['forms'][fm]['full_wmae'],
            pc['random']['forms'][fm]['full_wmae']))
    print('%-30s %-14s %-14s %-14s' % (NAME[fm], *cells))


## Fig B.1 — 평가 변형 비교


In [ ]:
run('run_fig_hybrid_variants.py', '(Fig B.1)')


---
## 수치 감사 — 재산출값 vs 원고

그림·표 안에 박힌 수치가 본문을 따라오지 않아 생긴 불일치가 실제로 여러 번 있었다 (Table 3 SC 열, Fig 10 범례 `89 pts / 4.1%`, Fig 7 상자 `0.5–1.2%`). 아래 셀은 **지금 데이터로 다시 계산한 값**과 **`.tex` 에 인쇄된 값**을 나란히 놓는다.

`TEX` 경로는 원고 저장소를 가리키도록 바꿔 쓴다.


In [ ]:
import io
import re
import numpy as np
import jeet_acloss_rbf.RbfModelBuilder as RB
from jeet_acloss_rbf.AcLossJsonReader import AcLossJsonReader
from jeet_acloss_rbf.pipeline import AcLossPipeline, DEFAULT_CONFIG

TEX = os.environ.get('JEET_TEX',
                     r'E:\KDH\Overleaf\JEET-2024_rev1\JEET_KDH_10p.tex')
BIG = 1e9


def wmae(e, w):
    return float(np.sum(np.abs(e) * w) / np.sum(w))


rows = []
for scale in ('Ref', 'HalfSC', 'SC'):
    path = os.path.join(DEFAULT_CONFIG['data_root'],
                        DEFAULT_CONFIG['json'][scale])
    recs, err = AcLossJsonReader.read(path, scale)
    assert err is None, err
    m = AcLossPipeline().build_model(scale)
    pool = RB.match_records_and_create_dataset(recs, 50.0, 0.3, 3.0)
    ds = RB.match_records_and_create_dataset(recs, 50.0, 0.0, BIG)
    af = m.predict(ds.speeds_k * 1000.0, ds.irms_arr, ds.phase_arr)
    e = (ds.h_ac_arr * af - ds.f_ac_arr) / (ds.f_ac_arr + 1e-12) * 100
    r = (ds.h_ac_arr - ds.f_ac_arr) / (ds.f_ac_arr + 1e-12) * 100
    rows.append((scale, len(pool), len(ds),
                 wmae(r, ds.f_ac_arr), wmae(e, ds.f_ac_arr)))

print('%-8s %6s %6s %10s %10s' % ('model', '후보', '평가',
                                  '무보정%', '보정후%'))
for s, np_, ne, r, w in rows:
    print('%-8s %6d %6d %10.1f %10.2f' % (s, np_, ne, r, w))

tex = io.open(TEX, encoding='utf-8', errors='replace').read() \
    if os.path.exists(TEX) else ''
if tex:
    lo, hi = min(x[4] for x in rows), max(x[4] for x in rows)
    rlo, rhi = min(x[3] for x in rows), max(x[3] for x in rows)
    want = '%.0f--%.0f' % (round(rlo), round(rhi))
    want2 = '%.1f--%.1f' % (lo, hi)
    print()
    print('원고 헤드라인 존재 확인:')
    for pat in (want, want2):
        print('   %-12s %s' % (pat, '있음' if pat in tex else '>>> 불일치'))
    for s, np_, ne, r, w in rows:
        tag = '%.2f' % w
        print('   %-8s %-6s %s' % (s, tag,
              '있음' if tag in tex else '(본문 미인쇄 — 확인)'))
else:
    print('\n[!] TEX 경로 없음 — JEET_TEX 로 지정하면 대조까지 한다')
